# Phase 4 — GenAI Enhancement & XAI Recovery (RQ3)

**Objective:** Apply image-restoration to the degraded test sets and measure how much **accuracy** *and* **XAI fidelity** are recovered, compared with a CLAHE preprocessing baseline.

**Methods:**
- **CLAHE** baseline — classical contrast-limited adaptive histogram equalization (per-channel on Lab).
- **GenAI** — pretrained Real-ESRGAN / SwinIR style restoration through HuggingFace `diffusers`/`transformers`. Falls back automatically to a small denoising autoencoder if the heavy model fails to load (e.g. Colab disk pressure).

**Outputs (under `Drive/Thesis/results/phase4_genai_enhancement/`):**
- `metrics/recovery_accuracy.csv`  — model × degradation × level × {raw, clahe, genai} accuracy
- `metrics/recovery_xai.csv`       — model × method × degradation × level × {raw, clahe, genai} mean stability/insertion
- `samples/recovery_<id>.png`      — clean / degraded / clahe / genai side-by-side
- `plots/recovery_accuracy_<type>.png`
- `plots/recovery_xai_<metric>_<type>.png`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%pip install -q timm diffusers transformers accelerate scikit-image

## Config

In [ ]:
from pathlib import Path
import torch

DRIVE_ROOT      = Path('/content/drive/MyDrive/Thesis')
RESULTS_ROOT    = DRIVE_ROOT / 'results'
CHECKPOINTS_DIR = DRIVE_ROOT / 'checkpoints'
PHASE_DIR       = RESULTS_ROOT / 'phase4_genai_enhancement'
PHASE1_DIR      = RESULTS_ROOT / 'phase1_data_engineering'
PHASE2_DIR      = RESULTS_ROOT / 'phase2_model_benchmarking'
for sub in ('metrics', 'plots', 'samples', 'logs'):
    (PHASE_DIR / sub).mkdir(parents=True, exist_ok=True)

LOCAL_ROOT   = Path('/content/data')
APTOS_DIR    = LOCAL_ROOT / 'aptos'
DEGRADED_DIR = LOCAL_ROOT / 'degraded'
ENHANCED_DIR = LOCAL_ROOT / 'enhanced'
ENHANCED_DIR.mkdir(parents=True, exist_ok=True)

APTOS_IMAGES = next(p for p in APTOS_DIR.rglob('train_images') if p.is_dir())
PRISTINE_CSV = PHASE1_DIR / 'metrics' / 'pristine_split.csv'

NUM_CLASSES = 5
IMAGE_SIZE  = 224
MODEL_NAMES = ('resnet50', 'efficientnet_b3', 'vit_base')
TIMM_NAMES  = {'resnet50': 'resnet50', 'efficientnet_b3': 'efficientnet_b3',
               'vit_base': 'vit_base_patch16_224'}
DEGRADATION_TYPES  = ('blur', 'exposure', 'noise')
DEGRADATION_LEVELS = ('low', 'mid', 'high')
ENHANCERS = ('clahe', 'genai')

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Device:', DEVICE)

## Enhancer 1 — CLAHE baseline

In [ ]:
import cv2
import numpy as np
from PIL import Image

_clahe = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8, 8))

def enhance_clahe(img: Image.Image) -> Image.Image:
    arr = np.array(img.convert('RGB'))
    lab = cv2.cvtColor(arr, cv2.COLOR_RGB2LAB)
    lab[..., 0] = _clahe.apply(lab[..., 0])
    return Image.fromarray(cv2.cvtColor(lab, cv2.COLOR_LAB2RGB))

## Enhancer 2 — GenAI restoration

Primary: HuggingFace `caidas/swin2SR-classical-sr-x2-64` (Swin Image Restoration). If unavailable, falls back to a tiny U-Net trained on-the-fly from the pristine set (handful of epochs).

In [ ]:
import torch.nn as nn
import torch.nn.functional as F

GENAI_BACKEND = None

def _try_swinir():
    """Try to load a HuggingFace Swin2SR pipeline. Returns (process_fn) or None."""
    try:
        from transformers import Swin2SRForImageSuperResolution, Swin2SRImageProcessor
        proc  = Swin2SRImageProcessor.from_pretrained('caidas/swin2SR-classical-sr-x2-64')
        model = Swin2SRForImageSuperResolution.from_pretrained('caidas/swin2SR-classical-sr-x2-64').to(DEVICE).eval()
        @torch.no_grad()
        def process(img: Image.Image) -> Image.Image:
            inputs = proc(img, return_tensors='pt').to(DEVICE)
            out = model(**inputs).reconstruction
            arr = out.squeeze().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
            return Image.fromarray((arr * 255).astype(np.uint8)).resize(img.size)
        return process
    except Exception as e:
        print('[swinir failed]', e)
        return None

class TinyUNet(nn.Module):
    def __init__(self, ch=32):
        super().__init__()
        def conv(i, o): return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.ReLU(inplace=True),
                                             nn.Conv2d(o, o, 3, padding=1), nn.ReLU(inplace=True))
        self.e1 = conv(3, ch); self.e2 = conv(ch, ch*2); self.e3 = conv(ch*2, ch*4)
        self.up2 = nn.ConvTranspose2d(ch*4, ch*2, 2, stride=2)
        self.d2 = conv(ch*4, ch*2)
        self.up1 = nn.ConvTranspose2d(ch*2, ch, 2, stride=2)
        self.d1 = conv(ch*2, ch)
        self.out = nn.Conv2d(ch, 3, 1)
    def forward(self, x):
        e1 = self.e1(x); e2 = self.e2(F.max_pool2d(e1, 2)); e3 = self.e3(F.max_pool2d(e2, 2))
        d2 = self.d2(torch.cat([self.up2(e3), e2], dim=1))
        d1 = self.d1(torch.cat([self.up1(d2), e1], dim=1))
        return torch.sigmoid(self.out(d1))

def _train_fallback_unet(pristine_csv, images_dir, ckpt_path, n_epochs=2, n_train=400):
    import pandas as pd
    from torchvision import transforms as T
    df = pd.read_csv(pristine_csv).sample(n_train, random_state=0)
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    model = TinyUNet().to(DEVICE)
    optim = torch.optim.AdamW(model.parameters(), lr=1e-3)
    crit  = nn.L1Loss()
    for ep in range(n_epochs):
        np.random.shuffle(df.values)
        for i, row in enumerate(df.itertuples()):
            img = Image.open(images_dir / f'{row.id_code}.png').convert('RGB')
            target = tfm(img).unsqueeze(0).to(DEVICE)
            # synth corruption: Gaussian noise OR blur (random)
            if np.random.rand() < 0.5:
                noisy = (target + 0.1 * torch.randn_like(target)).clamp(0, 1)
            else:
                k = 5
                w = torch.ones((3, 1, k, k), device=DEVICE) / (k * k)
                noisy = F.conv2d(target, w, padding=k//2, groups=3)
            optim.zero_grad(); loss = crit(model(noisy), target); loss.backward(); optim.step()
    torch.save(model.state_dict(), ckpt_path)
    return model

def _make_unet_processor(model):
    from torchvision import transforms as T
    tfm = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor()])
    @torch.no_grad()
    def process(img: Image.Image) -> Image.Image:
        x = tfm(img).unsqueeze(0).to(DEVICE)
        out = model(x).squeeze().clamp(0, 1).cpu().permute(1, 2, 0).numpy()
        return Image.fromarray((out * 255).astype(np.uint8)).resize(img.size)
    return process

enhance_genai = _try_swinir()
if enhance_genai is None:
    print('Falling back to TinyUNet trained on pristine.')
    ckpt = CHECKPOINTS_DIR / 'genai_unet.pt'
    unet = TinyUNet().to(DEVICE)
    if ckpt.exists():
        unet.load_state_dict(torch.load(ckpt, map_location=DEVICE))
    else:
        unet = _train_fallback_unet(PRISTINE_CSV, APTOS_IMAGES, ckpt)
    enhance_genai = _make_unet_processor(unet)
    GENAI_BACKEND = 'unet'
else:
    GENAI_BACKEND = 'swin2sr'
print('GenAI backend:', GENAI_BACKEND)

## Build enhanced versions of every degraded test image

We restrict to the held-out test ids from Phase 2 so accuracy is comparable.

In [ ]:
import pandas as pd
from tqdm.auto import tqdm

test_ids = set(pd.read_csv(PHASE2_DIR / 'metrics' / 'test_ids.csv')['id_code'].astype(str))

def build_enhanced(method_name, fn):
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            src_dir = DEGRADED_DIR / k / l
            out_dir = ENHANCED_DIR / method_name / k / l
            out_dir.mkdir(parents=True, exist_ok=True)
            mani_in  = pd.read_csv(src_dir / 'manifest.csv')
            mani_in  = mani_in[mani_in['id_code'].astype(str).isin(test_ids)]
            rows = []
            for _, row in tqdm(mani_in.iterrows(), total=len(mani_in),
                                desc=f'{method_name}/{k}/{l}', leave=False):
                src = src_dir / row['rel_path']
                out = out_dir / row['rel_path']
                if not out.exists():
                    fn(Image.open(src).convert('RGB')).save(out)
                rows.append({**row.to_dict(), 'method': method_name, 'rel_path': row['rel_path']})
            pd.DataFrame(rows).to_csv(out_dir / 'manifest.csv', index=False)

build_enhanced('clahe', enhance_clahe)
build_enhanced('genai', enhance_genai)
print('Enhanced sets built under', ENHANCED_DIR)

## Re-evaluate the three classifiers on enhanced sets

In [ ]:
import timm
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms as T
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score

TFM_EVAL = T.Compose([T.Resize((IMAGE_SIZE, IMAGE_SIZE)), T.ToTensor(),
                       T.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])])

class FolderDataset(Dataset):
    def __init__(self, root):
        self.root = Path(root); self.df = pd.read_csv(self.root / 'manifest.csv')
    def __len__(self): return len(self.df)
    def __getitem__(self, i):
        row = self.df.iloc[i]
        return TFM_EVAL(Image.open(self.root / row['rel_path']).convert('RGB')), int(row['diagnosis']), row['id_code']

@torch.no_grad()
def evaluate(model, loader):
    model.eval(); ys, ps, probs = [], [], []
    for x, y, _ in loader:
        prob = torch.softmax(model(x.to(DEVICE)), 1).cpu().numpy()
        probs.append(prob); ps.append(prob.argmax(1)); ys.append(y.numpy())
    y = np.concatenate(ys); p = np.concatenate(ps); pr = np.concatenate(probs)
    out = {'accuracy': accuracy_score(y, p), 'f1_macro': f1_score(y, p, average='macro')}
    try: out['auc_macro_ovr'] = roc_auc_score(y, pr, average='macro', multi_class='ovr')
    except ValueError: out['auc_macro_ovr'] = float('nan')
    return out

def build_model(name):
    m = timm.create_model(TIMM_NAMES[name], pretrained=False, num_classes=NUM_CLASSES)
    m._dr_arch = name; return m

rows = []
for name in MODEL_NAMES:
    print(f'\n=== {name} ===')
    model = build_model(name).to(DEVICE)
    state = torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)
    model.load_state_dict(state['state_dict'])
    for k in DEGRADATION_TYPES:
        for l in DEGRADATION_LEVELS:
            for variant in ('raw', *ENHANCERS):
                if variant == 'raw':
                    root = DEGRADED_DIR / k / l
                else:
                    root = ENHANCED_DIR / variant / k / l
                ds = FolderDataset(root)
                ds.df = ds.df[ds.df['id_code'].astype(str).isin(test_ids)].reset_index(drop=True)
                loader = DataLoader(ds, batch_size=32, num_workers=2, pin_memory=True)
                m = evaluate(model, loader)
                rows.append({'model': name, 'degradation': k, 'level': l, 'variant': variant, **m})
                print(f'  {k}/{l}/{variant}: acc={m["accuracy"]:.4f}')
    del model; torch.cuda.empty_cache()

rec_df = pd.DataFrame(rows)
rec_csv = PHASE_DIR / 'metrics' / 'recovery_accuracy.csv'
rec_df.to_csv(rec_csv, index=False)
print('\nSaved:', rec_csv)
rec_df.head()

## Recovery plots — accuracy

In [ ]:
import matplotlib.pyplot as plt
level_order = ['low', 'mid', 'high']
for kind in DEGRADATION_TYPES:
    fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2 * len(MODEL_NAMES), 4), sharey=True)
    for ax, name in zip(axes, MODEL_NAMES):
        sub = rec_df[(rec_df['degradation'] == kind) & (rec_df['model'] == name)].copy()
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        for variant in ('raw', 'clahe', 'genai'):
            d = sub[sub['variant'] == variant].sort_values('level')
            ax.plot(d['level'], d['accuracy'], marker='o', label=variant)
        ax.set_title(name); ax.set_ylim(0, 1); ax.grid(alpha=0.3)
        ax.set_xlabel('level')
    axes[0].set_ylabel('accuracy'); axes[-1].legend()
    fig.suptitle(f'Accuracy recovery — {kind}', fontsize=13)
    plt.tight_layout()
    out = PHASE_DIR / 'plots' / f'recovery_accuracy_{kind}.png'
    plt.savefig(out, dpi=150); plt.show()
    print('Saved:', out)

## Re-run a slim XAI evaluation on enhanced images

For speed we only re-compute **stability** (Spearman vs clean heatmap) and **insertion AUC** on a subset of test images, for the cheapest method per model (Grad-CAM for CNNs, Attention for ViT).

In [ ]:
import torch.nn.functional as F
from scipy.stats import spearmanr

def get_xai_target_layer(model):
    arch = model._dr_arch
    if arch == 'resnet50':        return model.layer4[-1]
    if arch == 'efficientnet_b3': return model.blocks[-1]
    if arch == 'vit_base':        return model.blocks[-1].norm1

def gradcam_heatmap(model, x, target_class):
    layer = get_xai_target_layer(model)
    acts, grads = {}, {}
    h1 = layer.register_forward_hook(lambda _, __, o: acts.update(v=o.detach()))
    h2 = layer.register_full_backward_hook(lambda _, gi, go: grads.update(v=go[0].detach()))
    try:
        model.zero_grad(set_to_none=True)
        x = x.clone().requires_grad_(True)
        logits = model(x); logits[0, target_class].backward()
        a, g = acts['v'][0], grads['v'][0]
        cam = torch.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
        cam -= cam.min()
        if cam.max() > 0: cam /= cam.max()
        cam = F.interpolate(cam[None, None], size=x.shape[-2:], mode='bilinear', align_corners=False)
        return cam.squeeze().cpu().numpy()
    finally:
        h1.remove(); h2.remove()

@torch.no_grad()
def attention_rollout(model, x):
    attns = []; handles = []
    for blk in model.blocks:
        attn_module = blk.attn
        def make_hook(store, mod):
            def _h(module, inp, out):
                B, N, C = inp[0].shape
                qkv = mod.qkv(inp[0]).reshape(B, N, 3, mod.num_heads, C // mod.num_heads).permute(2, 0, 3, 1, 4)
                q, k, _ = qkv[0], qkv[1], qkv[2]
                a = (q @ k.transpose(-2, -1)) * mod.scale
                store.append(a.softmax(dim=-1).detach())
            return _h
        handles.append(attn_module.register_forward_hook(make_hook(attns, attn_module)))
    try: _ = model(x)
    finally:
        for h in handles: h.remove()
    res = None
    for a in attns:
        a = a.mean(dim=1) + torch.eye(a.size(-1), device=a.device)
        a = a / a.sum(dim=-1, keepdim=True)
        res = a if res is None else a @ res
    cls_p = res[0, 0, 1:]
    side = int(cls_p.numel() ** 0.5)
    heat = F.interpolate(cls_p.reshape(side, side)[None, None],
                          size=x.shape[-2:], mode='bilinear', align_corners=False)
    h = heat.squeeze().cpu().numpy()
    if h.max() > h.min(): h = (h - h.min()) / (h.max() - h.min())
    return h

@torch.no_grad()
def insertion_auc(model, x, heatmap, target_class, steps=20):
    H, W = heatmap.shape
    order = np.argsort(-heatmap.flatten())
    chunk = max(1, len(order) // steps)
    base = F.avg_pool2d(x, 15, stride=1, padding=7)
    probs = []
    for s in range(steps + 1):
        idx = order[: s * chunk]
        rows, cols = np.unravel_index(idx, (H, W))
        x_mod = base.clone(); x_mod[0, :, rows, cols] = x[0, :, rows, cols]
        probs.append(torch.softmax(model(x_mod), 1)[0, target_class].item())
    return float(np.trapz(probs, dx=1 / steps))

def stab(a, b):
    rho, _ = spearmanr(a.flatten(), b.flatten())
    return float(rho) if rho == rho else 0.0

In [ ]:
labels_df = pd.read_csv(PRISTINE_CSV).set_index('id_code')['diagnosis']
EXPLAIN_IDS = list(test_ids)[:15]   # small subset

def load_tensor(path):
    return TFM_EVAL(Image.open(path).convert('RGB')).unsqueeze(0).to(DEVICE)

METHOD_FOR = {'resnet50': ('gradcam', gradcam_heatmap),
              'efficientnet_b3': ('gradcam', gradcam_heatmap),
              'vit_base': ('attention', attention_rollout)}

rows = []
for name in MODEL_NAMES:
    method_name, fn = METHOD_FOR[name]
    model = build_model(name).to(DEVICE)
    model.load_state_dict(torch.load(CHECKPOINTS_DIR / f'{name}_best.pt', map_location=DEVICE)['state_dict'])
    model.eval()
    print(f'\n=== {name} / {method_name} ===')
    for id_code in tqdm(EXPLAIN_IDS, desc=name):
        if id_code not in labels_df.index: continue
        label = int(labels_df[id_code])
        x_clean = load_tensor(APTOS_IMAGES / f'{id_code}.png')
        try: h_clean = fn(model, x_clean, label)
        except Exception as e: print('  skip clean:', e); continue
        for k in DEGRADATION_TYPES:
            for l in DEGRADATION_LEVELS:
                for variant in ('raw', *ENHANCERS):
                    src = (DEGRADED_DIR if variant == 'raw' else ENHANCED_DIR / variant) / k / l / f'{id_code}.png'
                    if not src.exists(): continue
                    x = load_tensor(src)
                    try: h = fn(model, x, label)
                    except Exception: continue
                    rows.append({'id_code': id_code, 'model': name, 'method': method_name,
                                 'degradation': k, 'level': l, 'variant': variant,
                                 'stability': stab(h_clean, h),
                                 'insertion_auc': insertion_auc(model, x, h, label)})
    del model; torch.cuda.empty_cache()

xai_rec = pd.DataFrame(rows)
out_csv = PHASE_DIR / 'metrics' / 'recovery_xai.csv'
xai_rec.to_csv(out_csv, index=False)
print('Saved:', out_csv)
xai_rec.head()

## Recovery plots — XAI

In [ ]:
for metric in ('stability', 'insertion_auc'):
    for kind in DEGRADATION_TYPES:
        sub = xai_rec[xai_rec['degradation'] == kind].copy()
        if sub.empty: continue
        sub['level'] = pd.Categorical(sub['level'], categories=level_order, ordered=True)
        g = sub.groupby(['model', 'variant', 'level'])[metric].mean().reset_index()
        fig, axes = plt.subplots(1, len(MODEL_NAMES), figsize=(4.2 * len(MODEL_NAMES), 4), sharey=True)
        for ax, name in zip(axes, MODEL_NAMES):
            for variant in ('raw', 'clahe', 'genai'):
                d = g[(g['model'] == name) & (g['variant'] == variant)].sort_values('level')
                ax.plot(d['level'], d[metric], marker='o', label=variant)
            ax.set_title(name); ax.grid(alpha=0.3); ax.set_xlabel('level')
        axes[0].set_ylabel(metric); axes[-1].legend()
        fig.suptitle(f'XAI {metric} recovery — {kind}', fontsize=13)
        plt.tight_layout()
        out = PHASE_DIR / 'plots' / f'recovery_xai_{metric}_{kind}.png'
        plt.savefig(out, dpi=150); plt.show()

## Qualitative side-by-side: clean / degraded / clahe / genai

In [ ]:
demo_id = EXPLAIN_IDS[0]
fig, axes = plt.subplots(len(DEGRADATION_TYPES), 4, figsize=(13, 3.2 * len(DEGRADATION_TYPES)))
for r, kind in enumerate(DEGRADATION_TYPES):
    clean = Image.open(APTOS_IMAGES / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    deg   = Image.open(DEGRADED_DIR / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    cl    = Image.open(ENHANCED_DIR / 'clahe' / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    gn    = Image.open(ENHANCED_DIR / 'genai' / kind / 'high' / f'{demo_id}.png').convert('RGB').resize((IMAGE_SIZE, IMAGE_SIZE))
    for ax, im, ttl in zip(axes[r], (clean, deg, cl, gn), ('clean', f'{kind}-high', 'CLAHE', 'GenAI')):
        ax.imshow(im); ax.set_title(ttl); ax.axis('off')
plt.tight_layout()
out = PHASE_DIR / 'samples' / f'recovery_{demo_id}.png'
plt.savefig(out, dpi=150, bbox_inches='tight'); plt.show()
print('Saved:', out)

---
**Phase 4 complete.** Proceed to `05_phase5_quality_aware_ensemble.ipynb`.